In [3]:
#clone the git repository
!git clone https://github.com/malsha890/NLP-Based-Singlish-Need-Classification-System-.git
%cd NLP-Based-Singlish-Need-Classification-System-/"nlp service"

fatal: destination path 'NLP-Based-Singlish-Need-Classification-System-' already exists and is not an empty directory.
/content/NLP-Based-Singlish-Need-Classification-System-/nlp service


In [6]:
#install required libraries
!pip install transformers peft

In [4]:
#Build the hybrid tokenizer, based on XLM-R's own vocabulary
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("xlm-roberta-base")
MAX_LEN = 64

def hybrid_encode(text, oov_threshold=0.4):
    ids = [tokenizer.bos_token_id]
    for word in text.strip().split():
        pieces = tokenizer.tokenize(word)
        fragmentation = len(pieces) / max(len(word), 1)
        if fragmentation > oov_threshold or tokenizer.unk_token in pieces:
            # character fallback, using XLM-R's own single-character tokens
            for ch in word:
                ch_ids = tokenizer.convert_tokens_to_ids(tokenizer.tokenize(ch))
                ids.extend(ch_ids if ch_ids else [tokenizer.unk_token_id])
        else:
            ids.extend(tokenizer.convert_tokens_to_ids(pieces))
    ids.append(tokenizer.eos_token_id)
    ids = ids[:MAX_LEN]
    attention_mask = [1] * len(ids)
    pad_len = MAX_LEN - len(ids)
    ids += [tokenizer.pad_token_id] * pad_len
    attention_mask += [0] * pad_len
    return ids, attention_mask

def baseline_encode(text):
    out = tokenizer(text, padding="max_length", truncation=True, max_length=MAX_LEN)
    return out["input_ids"], out["attention_mask"]

In [5]:
#Auxiliary features
NEED_KEYWORDS = ["one", "epa", "please", "ikmanin", "ikmanata", "help",
                  "urgent", "one", "puluwanda", "asaneepa"]

def get_aux_features(text):
    words = text.lower().split()
    length_feature = min(len(words) / 20, 1.0)  # normalised, capped at 1.0
    keyword_hits = sum(1 for w in words if w in NEED_KEYWORDS)
    keyword_feature = min(keyword_hits / 3, 1.0)
    return [length_feature, keyword_feature]

In [6]:
#Dataset class
import torch
from torch.utils.data import Dataset

CATEGORIES = ["medical aid", "shelter", "food/water", "rescue/missing", "other"]
LABEL2ID = {c: i for i, c in enumerate(CATEGORIES)}

class NeedDataset(Dataset):
    def __init__(self, dataframe, encode_fn):
        self.df = dataframe.reset_index(drop=True)
        self.encode_fn = encode_fn

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        input_ids, attention_mask = self.encode_fn(row["text"])
        aux = get_aux_features(row["text"])
        return {
            "input_ids": torch.tensor(input_ids),
            "attention_mask": torch.tensor(attention_mask),
            "aux_features": torch.tensor(aux, dtype=torch.float),
            "label": torch.tensor(LABEL2ID[row["category"]]),
        }

In [7]:
!pip install -U torchao

In [1]:
#The model class
from transformers import AutoModel
from peft import LoraConfig, get_peft_model
import torch.nn as nn

class SinglishNeedClassifier(nn.Module):
    def __init__(self, num_categories=5, aux_feature_dim=2):
        super().__init__()
        base = AutoModel.from_pretrained("xlm-roberta-base")
        lora_config = LoraConfig(
            r=8, lora_alpha=16,
            target_modules=["query", "value"],
            lora_dropout=0.1,
        )
        self.encoder = get_peft_model(base, lora_config)
        hidden = base.config.hidden_size
        self.classifier = nn.Sequential(
            nn.Linear(hidden + aux_feature_dim, 128),
            nn.ReLU(), nn.Dropout(0.2),
            nn.Linear(128, num_categories),
        )

    def forward(self, input_ids, attention_mask, aux_features):
        out = self.encoder(input_ids=input_ids, attention_mask=attention_mask)
        pooled = out.last_hidden_state[:, 0, :]
        combined = torch.cat([pooled, aux_features], dim=1)
        return self.classifier(combined)

In [8]:
#The small subset sanity check
import pandas as pd
from torch.utils.data import DataLoader
from torch.optim import AdamW

df = pd.read_csv("../data/train_augmented.csv").sample(40, random_state=1)  # tiny subset
dataset = NeedDataset(df, hybrid_encode)
loader = DataLoader(dataset, batch_size=8, shuffle=True)

model = SinglishNeedClassifier().to("cuda")
optimizer = AdamW(model.parameters(), lr=2e-5)
criterion = nn.CrossEntropyLoss()

model.train()
for epoch in range(2):
    for batch in loader:
        optimizer.zero_grad()
        logits = model(batch["input_ids"].to("cuda"),
                        batch["attention_mask"].to("cuda"),
                        batch["aux_features"].to("cuda"))
        loss = criterion(logits, batch["label"].to("cuda"))
        loss.backward()
        optimizer.step()
    print(f"Epoch {epoch}, last batch loss: {loss.item():.4f}")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] XLMRobertaModel LOAD REPORT from: xlm-roberta-base
Key                       | Status     |  | 
--------------------------+------------+--+-
lm_head.dense.weight      | UNEXPECTED |  | 
lm_head.layer_norm.bias   | UNEXPECTED |  | 
lm_head.dense.bias        | UNEXPECTED |  | 
lm_head.bias              | UNEXPECTED |  | 
lm_head.layer_norm.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Epoch 0, last batch loss: 1.5608
Epoch 1, last batch loss: 1.5839


In [9]:
#print the average loss across the whole epoch
model.train()
for epoch in range(4):  # bump to 4, a bit more signal
    total_loss = 0
    for batch in loader:
        optimizer.zero_grad()
        logits = model(batch["input_ids"].to("cuda"),
                        batch["attention_mask"].to("cuda"),
                        batch["aux_features"].to("cuda"))
        loss = criterion(logits, batch["label"].to("cuda"))
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    print(f"Epoch {epoch}, average loss: {total_loss/len(loader):.4f}")

Epoch 0, average loss: 1.5965
Epoch 1, average loss: 1.6132
Epoch 2, average loss: 1.6195
Epoch 3, average loss: 1.6098


In [10]:
#Test that Can It Overfit a Tiny, Repeated Batch
model = SinglishNeedClassifier().to("cuda")  # fresh model
optimizer = AdamW(model.parameters(), lr=2e-5)
criterion = nn.CrossEntropyLoss()

tiny_batch = next(iter(loader))  # just one batch, 8 examples, reused

model.train()
for step in range(50):
    optimizer.zero_grad()
    logits = model(tiny_batch["input_ids"].to("cuda"),
                    tiny_batch["attention_mask"].to("cuda"),
                    tiny_batch["aux_features"].to("cuda"))
    loss = criterion(logits, tiny_batch["label"].to("cuda"))
    loss.backward()
    optimizer.step()
    if step % 10 == 0:
        print(f"step {step}, loss: {loss.item():.4f}")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] XLMRobertaModel LOAD REPORT from: xlm-roberta-base
Key                       | Status     |  | 
--------------------------+------------+--+-
lm_head.dense.weight      | UNEXPECTED |  | 
lm_head.layer_norm.bias   | UNEXPECTED |  | 
lm_head.dense.bias        | UNEXPECTED |  | 
lm_head.bias              | UNEXPECTED |  | 
lm_head.layer_norm.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


step 0, loss: 1.5751
step 10, loss: 1.5692
step 20, loss: 1.4807
step 30, loss: 1.5718
step 40, loss: 1.4907


In [11]:
#Check 1: Are the LoRA Parameters Actually Trainable
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())
print(f"Trainable: {trainable:,} / Total: {total:,} ({100*trainable/total:.2f}%)")

Trainable: 394,245 / Total: 278,437,893 (0.14%)


In [12]:
#check 2:are gradients actually reaching those trainable parameters
optimizer.zero_grad()
logits = model(tiny_batch["input_ids"].to("cuda"),
                tiny_batch["attention_mask"].to("cuda"),
                tiny_batch["aux_features"].to("cuda"))
loss = criterion(logits, tiny_batch["label"].to("cuda"))
loss.backward()

for name, param in model.classifier.named_parameters():
    if param.grad is not None:
        print(f"{name}: grad norm = {param.grad.norm().item():.6f}")
    else:
        print(f"{name}: NO GRADIENT")

0.weight: grad norm = 2.453706
0.bias: grad norm = 0.111360
3.weight: grad norm = 0.978367
3.bias: grad norm = 0.209049


In [13]:
#check 3
model = SinglishNeedClassifier().to("cuda")  # fresh model, undo previous 50 steps
optimizer = AdamW(model.parameters(), lr=2e-4)  # 10x higher
criterion = nn.CrossEntropyLoss()

model.train()
for step in range(50):
    optimizer.zero_grad()
    logits = model(tiny_batch["input_ids"].to("cuda"),
                    tiny_batch["attention_mask"].to("cuda"),
                    tiny_batch["aux_features"].to("cuda"))
    loss = criterion(logits, tiny_batch["label"].to("cuda"))
    loss.backward()
    optimizer.step()
    if step % 10 == 0:
        print(f"step {step}, loss: {loss.item():.4f}")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] XLMRobertaModel LOAD REPORT from: xlm-roberta-base
Key                       | Status     |  | 
--------------------------+------------+--+-
lm_head.dense.weight      | UNEXPECTED |  | 
lm_head.layer_norm.bias   | UNEXPECTED |  | 
lm_head.dense.bias        | UNEXPECTED |  | 
lm_head.bias              | UNEXPECTED |  | 
lm_head.layer_norm.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


step 0, loss: 1.6699
step 10, loss: 1.4078
step 20, loss: 1.4488
step 30, loss: 1.2596
step 40, loss: 1.3423


In [14]:
#Check 4
model = SinglishNeedClassifier().to("cuda")
optimizer = AdamW(model.parameters(), lr=1e-3)  # another 5x higher
criterion = nn.CrossEntropyLoss()

model.train()
for step in range(100):  # more steps too
    optimizer.zero_grad()
    logits = model(tiny_batch["input_ids"].to("cuda"),
                    tiny_batch["attention_mask"].to("cuda"),
                    tiny_batch["aux_features"].to("cuda"))
    loss = criterion(logits, tiny_batch["label"].to("cuda"))
    loss.backward()
    optimizer.step()
    if step % 20 == 0:
        print(f"step {step}, loss: {loss.item():.4f}")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] XLMRobertaModel LOAD REPORT from: xlm-roberta-base
Key                       | Status     |  | 
--------------------------+------------+--+-
lm_head.dense.weight      | UNEXPECTED |  | 
lm_head.layer_norm.bias   | UNEXPECTED |  | 
lm_head.dense.bias        | UNEXPECTED |  | 
lm_head.bias              | UNEXPECTED |  | 
lm_head.layer_norm.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


step 0, loss: 1.5500
step 20, loss: 1.2630
step 40, loss: 0.5513
step 60, loss: 0.3245
step 80, loss: 0.0091


In [18]:
!git add .
!git commit -m " model class, sanity check, diagnostic tests, LR tuning done successfully"
!git push

hint: You've added another git repository inside your current repository.
hint: Clones of the outer repository will not contain the contents of
hint: the embedded repository and will not know how to obtain it.
hint: If you meant to add a submodule, use:
hint: 
hint: 	git submodule add <url> nlp service/NLP-Based-Singlish-Need-Classification-System-
hint: 
hint: If you added this path by mistake, you can remove it from the
hint: index with:
hint: 
hint: 	git rm --cached nlp service/NLP-Based-Singlish-Need-Classification-System-
hint: 
hint: See "git help submodule" for more information.
Author identity unknown

*** Please tell me who you are.

Run

  git config --global user.email "you@example.com"
  git config --global user.name "Your Name"

to set your account's default identity.
Omit --global to set the identity only in this repository.

fatal: unable to auto-detect email address (got 'root@1b5a2cca6055.(none)')
fatal: could not read Username for 'https://github.com': No such device 

In [19]:
!rm -rf NLP-Based-Singlish-Need-Classification-System-

!git clone https://github.com/malsha890/NLP-Based-Singlish-Need-Classification-System-.git
%cd NLP-Based-Singlish-Need-Classification-System-


Cloning into 'NLP-Based-Singlish-Need-Classification-System-'...
remote: Enumerating objects: 50, done.
remote: Counting objects: 100% (50/50), done.
remote: Compressing objects: 100% (39/39), done.
remote: Total 50 (delta 11), reused 43 (delta 4), pack-reused 0 (from 0)
Receiving objects: 100% (50/50), 197.01 KiB | 7.58 MiB/s, done.
Resolving deltas: 100% (11/11), done.
/content/NLP-Based-Singlish-Need-Classification-System-/nlp service/NLP-Based-Singlish-Need-Classification-System-
